In [1]:
import hoda
import tensorly as tl

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
tmin = 0
tmax=0.8
fmin=0.5
fmax = 16
sfreq = 32

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
    #BI2012(),
    #BI2013a(),
    #BI2014a(),
    #BI2014b(),
    #BI2015a(),
    #BI2015b(),
    BNCI2014_008(),
    #BNCI2014_009(),
    #BNCI2015_003(),
    #Cattan2019_VR(),
    #EPFLP300(),
    #Huebner2017(),
    #Huebner2018(),
    #Lee2019_ERP(),
    #Sosulski2019(),
]

evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    suffix="hoda_erp",
    overwrite=True,
    random_state=42,
    n_jobs=5,
)

<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.


To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.


/usr/local/lib/python3.10/dist-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(


In [7]:
from sklearn.pipeline import make_pipeline, Pipeline
from hoda.hoda import HODA, BTTDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from hoda.classification import Vectorize
from sklearn.feature_selection import SelectFwe
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import FunctionTransformer
import numpy as np
from hoda.tensorize import hankel_tensor

pipelines = dict()


pipelines['sLDA'] = make_pipeline(
        Vectorize(),
        LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

pipelines['HODA'] = GridSearchCV(
        Pipeline([
            ('hoda', HODA(
                max_iter=256,
                tol=1e-8,
                init ='svd',
                shrinkage='lw',
                toeplitz=None,
                obj='rt',
                solver='lanczos',
                taper=False,
                keep_train_info=False,
                verbose=False,
            )),
            ('vec', Vectorize()),
            ('lda', LinearDiscriminantAnalysis())
        ]),
        dict(hoda__rank=np.arange(8)+1),
        scoring='roc_auc',
    )

pipelines['BTTDA'] = Pipeline([
    ('bttda', BTTDA(
        max_blocks=16,
        hoda_params=dict(
            max_iter=256,
            tol=1e-8,
            init ='svd',
            shrinkage='lw',
            toeplitz=None,
            obj='rt',
            solver='lanczos',
            taper=False,
            keep_train_info=False,
            verbose=False,
            info_crit='bic',
        ),
        verbose=False,
        keep_train_info=False
    )),
    ('vec', Vectorize()),
    ('lda', LinearDiscriminantAnalysis())
])

pipelines['HODA_hankel'] = GridSearchCV(
        Pipeline([
            ('hankel', FunctionTransformer(hankel_tensor)),
            ('hoda', HODA(
                max_iter=256,
                tol=1e-8,
                init ='svd',
                shrinkage='lw',
                toeplitz=None,
                obj='rt',
                solver='lanczos',
                taper=False,
                keep_train_info=False,
                verbose=False,
            )),
            ('vec', Vectorize()),
            ('lda', LinearDiscriminantAnalysis())
        ]),
        dict(hoda__rank=np.arange(8)+1),
        scoring='roc_auc',
    )

pipelines['BTTDA_hankel'] = Pipeline([
    ('hankel', FunctionTransformer(hankel_tensor)),
    ('bttda', BTTDA(
        max_blocks=16,
        hoda_params=dict(
            max_iter=256,
            tol=1e-8,
            init ='svd',
            shrinkage='lw',
            toeplitz=None,
            obj='rt',
            solver='lanczos',
            taper=False,
            keep_train_info=False,
            verbose=False,
            info_crit='bic',
        ),
        verbose=False,
        keep_train_info=False
    )),
    ('vec', Vectorize()),
    ('lda', LinearDiscriminantAnalysis())
])

In [8]:
#import warnings
#warnings.filterwarnings("ignore")

results = evaluation.process(pipelines)

BNCI2014-008-WithinSession:   0%|                                                                                                       | 0/8 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
BNCI2014-008-WithinSession:   0%|                                                                                                       | 0/8 [00:07<?, ?it/s]


ValueError: 
All the 40 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 729, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/base.py", line 1152, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/pipeline.py", line 423, in fit
    Xt = self._fit(X, y, **fit_params_steps)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/pipeline.py", line 377, in _fit
    X, fitted_transformer = fit_transform_one_cached(
  File "/usr/local/lib/python3.10/dist-packages/joblib/memory.py", l

In [ ]:
results

In [ ]:
import seaborn as sns
order = results.groupby('pipeline')
order = order.score.aggregate('mean')
order = order.sort_values()

sns.barplot(
     data=results,
    y="score", x="dataset", hue="pipeline", hue_order=order.index,
)

In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
from moabb.analysis.plotting import meta_analysis_plot, paired_plot
_ = meta_analysis_plot(stats, 'HODA', 'BTTDA')
_  = paired_plot(results, 'HODA', 'BTTDA')